# NexTwin AI — Industrial Digital Twin Platform
## Notebook 02: Data Cleaning, Outlier Treatment, & Quality Scoring

### Objectives
1. **Handle Missing Values**: Apply domain-specific imputation (median for continuous, mode for categorical, interpolation for timeseries).
2. **Outlier Mitigation**: Identify outliers using Interquartile Range (IQR) and cap them rather than drop them (retaining operational extremes).
3. **De-duplication**: Detect and remove duplicate records.
4. **Consistency Checks**: Validate physics boundaries (e.g. temperatures > 0, utilization within [0, 100]).
5. **Column Normalization**: Clean column names to standard lowercase snake_case (removing spaces, special symbols, brackets).
6. **Data Quality Scoring**: Build a custom composite scoring system for monitoring dataset health.
7. **Export Clean Datasets**: Write the processed data files to `datasets/processed/` for downstream feature engineering.

In [1]:
import os
import pandas as pd
import numpy as np
import re

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Raw Datasets
We load all raw data tables for cleaning.

In [2]:
RAW_DIR = os.path.join("..", "datasets", "raw")
PROCESSED_DIR = os.path.join("..", "datasets", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Load datasets
df_pm = pd.read_csv(os.path.join(RAW_DIR, "ai4i_predictive_maintenance.csv"))
df_energy = pd.read_excel(os.path.join(RAW_DIR, "energy_consumption.csv"))
df_util = pd.read_csv(os.path.join(RAW_DIR, "machine_utilization.csv"))
df_prod = pd.read_csv(os.path.join(RAW_DIR, "production_metrics.csv"))
df_factory = pd.read_csv(os.path.join(RAW_DIR, "machine_sounds", "synthetic_factory_data.csv"))

# Load NASA Turbofan Train files
nasa_cols = ['unit_number', 'time_in_cycles', 'setting_1', 'setting_2', 'setting_3'] + [f'sensor_{i}' for i in range(1, 22)]
df_nasa_train_1 = pd.read_csv(os.path.join(RAW_DIR, "nasa_turbofan", "train_FD001.txt"), sep=r"\s+", header=None, names=nasa_cols)
df_nasa_test_1 = pd.read_csv(os.path.join(RAW_DIR, "nasa_turbofan", "test_FD001.txt"), sep=r"\s+", header=None, names=nasa_cols)
df_nasa_rul_1 = pd.read_csv(os.path.join(RAW_DIR, "nasa_turbofan", "RUL_FD001.txt"), sep=r"\s+", header=None, names=['RUL'])

print("All datasets loaded for cleaning.")

All datasets loaded for cleaning.


## 2. Normalize Column Names
Standardize names by turning them into lowercase snake_case, stripping units (like `[K]` or `[Nm]`), spaces, and brackets. This minimizes coding mistakes in downstream model training.

In [3]:
def normalize_column_names(df):
    new_cols = []
    for col in df.columns:
        col_str = str(col).strip()
        # Lowercase
        col_str = col_str.lower()
        # Remove bracketed units and parentheses
        col_str = re.sub(r'\[.*?\]', '', col_str)
        col_str = re.sub(r'\(.*?\)', '', col_str)
        # Replace spaces, dashes, slashes, and periods with underscore
        col_str = re.sub(r'[\s\-/\.]+', '_', col_str)
        # Remove consecutive underscores and trailing/leading underscores
        col_str = re.sub(r'_+', '_', col_str).strip('_')
        new_cols.append(col_str)
    df.columns = new_cols
    return df

# Normalize all datasets
df_pm = normalize_column_names(df_pm)
df_util = normalize_column_names(df_util)
df_prod = normalize_column_names(df_prod)
df_factory = normalize_column_names(df_factory)
df_nasa_train_1 = normalize_column_names(df_nasa_train_1)
df_nasa_test_1 = normalize_column_names(df_nasa_test_1)
df_nasa_rul_1 = normalize_column_names(df_nasa_rul_1)

# For energy dataset, map the abstract column names X1..X8, Y1, Y2 to descriptive domain-specific names
# X1 Relative Compactness, X2 Surface Area, X3 Wall Area, X4 Roof Area, X5 Overall Height,
# X6 Orientation, X7 Glazing Area, X8 Glazing Area Distribution, Y1 Heating Load, Y2 Cooling Load
energy_mapping = {
    'X1': 'relative_compactness', 'X2': 'surface_area', 'X3': 'wall_area', 'X4': 'roof_area',
    'X5': 'overall_height', 'X6': 'orientation', 'X7': 'glazing_area', 'X8': 'glazing_area_distribution',
    'Y1': 'heating_load', 'Y2': 'cooling_load'
}
df_energy = df_energy.rename(columns=energy_mapping)
df_energy = normalize_column_names(df_energy)

print("Normalized columns list sample:")
print("AI4I:", list(df_pm.columns))
print("Energy:", list(df_energy.columns))
print("Utilization:", list(df_util.columns))

Normalized columns list sample:
AI4I: ['udi', 'product_id', 'type', 'air_temperature', 'process_temperature', 'rotational_speed', 'torque', 'tool_wear', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf']
Energy: ['relative_compactness', 'surface_area', 'wall_area', 'roof_area', 'overall_height', 'orientation', 'glazing_area', 'glazing_area_distribution', 'heating_load', 'cooling_load']
Utilization: ['timestamp', 'machine_id', 'utilization_rate', 'operator_id', 'operational_status', 'energy_draw_kw']


## 3. Imputation and De-duplication
We handle any potential nulls and duplicates across the loaded datasets. For numeric variables, we use median imputation. For categorical, we use mode. Since our synthetic datasets are clean, this provides a safety framework for production.

In [4]:
def clean_dataset(df, name):
    print(f"Cleaning {name}...")
    initial_rows = len(df)
    
    # Remove duplicate rows
    df = df.drop_duplicates()
    dup_removed = initial_rows - len(df)
    if dup_removed > 0:
        print(f"Removed {dup_removed} duplicate rows from {name}")
        
    # Missing values imputation
    for col in df.columns:
        null_count = df[col].isnull().sum()
        if null_count > 0:
            print(f"Imputing {null_count} nulls in '{col}'")
            if pd.api.types.is_numeric_dtype(df[col]):
                # Median imputation for continuous variables
                df[col] = df[col].fillna(df[col].median())
            else:
                # Mode imputation for categorical variables
                df[col] = df[col].fillna(df[col].mode()[0])
    return df

df_pm = clean_dataset(df_pm, "AI4I")
df_energy = clean_dataset(df_energy, "Energy")
df_util = clean_dataset(df_util, "Utilization")
df_prod = clean_dataset(df_prod, "Production")
df_factory = clean_dataset(df_factory, "Factory Acoustic Sensor")
df_nasa_train_1 = clean_dataset(df_nasa_train_1, "NASA Turbofan FD001 Train")
df_nasa_test_1 = clean_dataset(df_nasa_test_1, "NASA Turbofan FD001 Test")

Cleaning AI4I...
Cleaning Energy...
Cleaning Utilization...
Cleaning Production...
Cleaning Factory Acoustic Sensor...
Cleaning NASA Turbofan FD001 Train...
Cleaning NASA Turbofan FD001 Test...


## 4. Outlier Detection and Capping
In industrial sensors, extreme outliers might signify critical states or failures. Rather than discarding these records (which removes valuable training examples for failure prediction), we perform IQR-based capping (winsorization) at the 1st and 99th percentiles.

In [5]:
def cap_outliers(df, columns_to_cap):
    df_capped = df.copy()
    for col in columns_to_cap:
        if col not in df_capped.columns:
            continue
        q1 = df_capped[col].quantile(0.01)
        q99 = df_capped[col].quantile(0.99)
        # Cap values
        df_capped[col] = np.clip(df_capped[col], q1, q99)
    return df_capped

# Target numerical columns for capping
pm_numeric = ['air_temperature', 'process_temperature', 'rotational_speed', 'torque', 'tool_wear']
energy_numeric = ['relative_compactness', 'surface_area', 'wall_area', 'roof_area', 'overall_height', 'heating_load', 'cooling_load']
factory_numeric = ['vibration_mm_s', 'temperature_c', 'pressure_bar', 'noise_level_db', 'sound_frequency_hz', 'sound_amplitude']

df_pm = cap_outliers(df_pm, pm_numeric)
df_energy = cap_outliers(df_energy, energy_numeric)
df_factory = cap_outliers(df_factory, factory_numeric)

print("Outlier capping completed for main continuous features.")

Outlier capping completed for main continuous features.


## 5. Physical Consistency Checks
We validate physical bounds of sensors:
1. Temperatures should be positive (in Kelvin or Celsius).
2. Utilization rate must be between 0 and 100%.
3. OEE must be between 0.0 and 1.1.
4. Rotational speed and Torque must be positive.

In [6]:
print("=== Performing Consistency Checks ===")

# AI4I checks
invalid_temp = df_pm[(df_pm['air_temperature'] <= 0) | (df_pm['process_temperature'] <= 0)]
print(f"AI4I invalid temperatures count: {len(invalid_temp)}")

# Utilization checks
invalid_util = df_util[(df_util['utilization_rate'] < 0) | (df_util['utilization_rate'] > 100)]
print(f"Utilization out of [0, 100] bounds count: {len(invalid_util)}")

# Production checks
invalid_oee = df_prod[(df_prod['oee'] < 0) | (df_prod['oee'] > 1.2)]
print(f"Production metrics out of OEE bounds count: {len(invalid_oee)}")

print("Consistency checks complete.")

=== Performing Consistency Checks ===
AI4I invalid temperatures count: 0
Utilization out of [0, 100] bounds count: 0
Production metrics out of OEE bounds count: 0
Consistency checks complete.


## 6. Data Quality Scoring Engine
We calculate a composite **Data Quality Score (DQS)** from 0 to 100 for each dataset. The score is computed as:

$$\text{DQS} = 100 \times \left(1 - \left(w_{\text{null}} \times \text{NullRate} + w_{\text{dup}} \times \text{DupRate} + w_{\text{outlier}} \times \text{OutlierRate}\right)\right)$$

Where weights are:
- $w_{\text{null}} = 0.5$ (high penalty for missing data)
- $w_{\text{dup}} = 0.2$ (penalty for duplicates)
- $w_{\text{outlier}} = 0.3$ (penalty for extreme observations beyond $1.5 \times \text{IQR}$)

In [7]:
def calculate_data_quality_score(df, name, numeric_cols):
    total_cells = df.size
    null_cells = df.isnull().sum().sum()
    null_rate = null_cells / total_cells if total_cells > 0 else 0
    
    dup_rows = df.duplicated().sum()
    dup_rate = dup_rows / len(df) if len(df) > 0 else 0
    
    outlier_cells = 0
    total_numeric_cells = 0
    for col in numeric_cols:
        if col in df.columns:
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outliers = ((df[col] < lower) | (df[col] > upper)).sum()
            outlier_cells += outliers
            total_numeric_cells += len(df)
            
    outlier_rate = outlier_cells / total_numeric_cells if total_numeric_cells > 0 else 0
    
    # Scoring weights
    w_null = 0.5
    w_dup = 0.2
    w_outlier = 0.3
    
    dqs = 100.0 * (1.0 - (w_null * null_rate + w_dup * dup_rate + w_outlier * outlier_rate))
    dqs = max(0.0, min(100.0, dqs))
    
    print(f"Data Quality Score for {name}: {dqs:.2f}/100")
    print(f"  - Null Rate: {null_rate*100:.3f}%")
    print(f"  - Duplicate Rate: {dup_rate*100:.3f}%")
    print(f"  - Outlier Rate: {outlier_rate*100:.3f}%")
    return dqs

print("--- Data Quality Review ---")
dqs_pm = calculate_data_quality_score(df_pm, "AI4I PM", pm_numeric)
dqs_energy = calculate_data_quality_score(df_energy, "Energy Consumption", energy_numeric)
dqs_factory = calculate_data_quality_score(df_factory, "Factory Sensor Data", factory_numeric)

--- Data Quality Review ---


Data Quality Score for AI4I PM: 99.75/100
  - Null Rate: 0.000%
  - Duplicate Rate: 0.000%
  - Outlier Rate: 0.836%
Data Quality Score for Energy Consumption: 100.00/100
  - Null Rate: 0.000%
  - Duplicate Rate: 0.000%
  - Outlier Rate: 0.000%


Data Quality Score for Factory Sensor Data: 99.82/100
  - Null Rate: 0.000%
  - Duplicate Rate: 0.000%
  - Outlier Rate: 0.584%


## 7. Save Cleaned Datasets
Now we write the normalized, cleaned, and consistency-checked datasets to the processed directory.

In [8]:
df_pm.to_csv(os.path.join(PROCESSED_DIR, "cleaned_ai4i_predictive_maintenance.csv"), index=False)
df_energy.to_csv(os.path.join(PROCESSED_DIR, "cleaned_energy_consumption.csv"), index=False)
df_util.to_csv(os.path.join(PROCESSED_DIR, "cleaned_machine_utilization.csv"), index=False)
df_prod.to_csv(os.path.join(PROCESSED_DIR, "cleaned_production_metrics.csv"), index=False)
df_factory.to_csv(os.path.join(PROCESSED_DIR, "cleaned_synthetic_factory_data.csv"), index=False)
df_nasa_train_1.to_csv(os.path.join(PROCESSED_DIR, "cleaned_nasa_train_fd001.csv"), index=False)
df_nasa_test_1.to_csv(os.path.join(PROCESSED_DIR, "cleaned_nasa_test_fd001.csv"), index=False)
df_nasa_rul_1.to_csv(os.path.join(PROCESSED_DIR, "cleaned_nasa_rul_fd001.csv"), index=False)

print("All processed datasets saved successfully to datasets/processed/.")

All processed datasets saved successfully to datasets/processed/.
